<a href="https://colab.research.google.com/github/mcgilmore/colab_alphapulldown/blob/main/colab_alphapulldown_AF3_AF2features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AlphaPulldown AF3 screen on Colab — reuse AF2 features
Updated 4 October 2026. Separate AF3 environment; existing AlphaPulldown `.pkl`/`.pkl.xz` protein objects provide alignments and templates. No sequence databases are downloaded or searched. This is AF3 inference with AF2-prepared features, not the native AF3 search protocol.

Select a fresh **A100 GPU runtime**. Run bootstrap alone and wait for its expected kernel restart. Then edit configuration and run cells in order. Start with one pair. Installation builds AF3 C++ components and can take several minutes; all commands stream their output.

Completed jobs are skipped on rerun. Interrupted AF3 jobs rerun from the beginning; there is no promise of sample-level recovery. Source commits, input hashes, sampling settings and weight hash are recorded on Drive. This notebook has been statically validated, but installation and inference have not been tested on a live Colab GPU.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'], check=True)
import condacolab
condacolab.install()  # Expected kernel restart; do not put more work in this cell.


## 1. Mount Drive and configure the screen
Create `baits.txt` with one bait name per line, and `preys.txt` with one candidate name per line. Names must match feature filenames, excluding `.pkl` or `.pkl.xz`. FASTA files are also accepted: accessions are read from headers and sequence lines are ignored. Simple whole-protein identifiers only: no residue/copy syntax here. Blank lines and lines beginning with `#` are ignored. Defaults use your `colab_ec` Drive folder. Set `PREY_LIST` to your actual candidate list or FASTA file, and `BAIT_LIST` to your fragment list/FASTA.

Use a new `RUN_NAME` when changing model settings, protein sequences/features, or software versions. Input changes within an existing run are refused to avoid mixing results.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, subprocess, hashlib, shutil, sys, time

drive.mount('/content/drive')
AP_REF = 'main'  # Resolved to a saved Git commit on first installation.
SOURCE_DIR = Path('/content/AlphaPulldown_AF3')
ENV_PREFIX = Path('/content/ap3_env')
FEATURE_DIR = Path('/content/drive/MyDrive/colab_ec/Escherichia_coli/features')
BAIT_LIST = Path('/content/drive/MyDrive/colab_ec/baits.txt')
PREY_LIST = Path('/content/drive/MyDrive/colab_ec/preys.txt')
OUTPUT_ROOT = Path('/content/drive/MyDrive/colab_ec/out')
RUN_NAME = '1b_K12_AF3'
MODEL_DIR = Path('/content/af3_models')
WEIGHTS_CACHE = Path('/content/drive/MyDrive/colab_ec/af3_weights')
NUM_RECYCLES = 10
NUM_SEEDS = 1
DIFFUSION_SAMPLES = 5
FLASH_ATTENTION = 'xla'  # Portable starting point; optional cudnn after verifying GPU support.
BASE_SEED = 42
PAIR_MSA = True  # Used for validation; AF3 translation handles paired/unpaired alignments.
START_JOB = 1  # 1-based inclusive, in the saved jobs.csv manifest.
END_JOB = 1    # Start with one pair; set None to run to the end.
CONTINUE_ON_ERROR = False

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Inherit Colab's CUDA library setup; let JAX allocate only what it needs.
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
CONDA = shutil.which('conda')
if not CONDA:
    raise RuntimeError('Run the CondaColab bootstrap, wait for restart, then resume here.')

def ap3_run(args, **kwargs):
    return subprocess.run([CONDA, 'run', '--no-capture-output', '-p', str(ENV_PREFIX), *args],
                          env=RUNTIME_ENV, check=True, **kwargs)
print('Run directory:', RUN_DIR)
if NUM_SEEDS < 1 or DIFFUSION_SAMPLES < 1 or NUM_RECYCLES < 1:
    raise ValueError('Seeds, samples and recycles must be positive.')


## 2. Install AF3 and pin the source revision
The first run resolves `AP_REF` to a commit and saves it in this run's Drive folder. Later sessions fetch that exact commit and its pinned submodules. This follows AlphaPulldown's source-install workflow so AF3 matches its patched backend. Use a new run name for software changes. AF2's environment is untouched.


In [ ]:
if not (ENV_PREFIX / 'bin/python').exists():
    print('Creating Python 3.11 environment with build tools…', flush=True)
    subprocess.run([CONDA, 'create', '-y', '-p', str(ENV_PREFIX), '-c', 'conda-forge',
                    '-c', 'bioconda', 'python=3.11', 'pip', 'kalign2', 'hmmer', 'hhsuite',
                    'libcifpp', 'sqlite', 'cmake', 'ninja', 'make', 'gcc_linux-64', 'gxx_linux-64'], check=True)
repo = 'https://github.com/KosinskiLab/AlphaPulldown.git'
revision_file = RUN_DIR / 'source_revision.json'
if revision_file.exists():
    source_record = json.loads(revision_file.read_text())
    if source_record['repository'] != repo:
        raise RuntimeError('Source repository differs from saved run.')
    revision = source_record['commit']
else:
    refs = subprocess.check_output(['git', 'ls-remote', repo, AP_REF, 'refs/heads/'+AP_REF,
                                    'refs/tags/'+AP_REF, 'refs/tags/'+AP_REF+'^{}'], text=True).splitlines()
    if not refs: raise RuntimeError(f'Cannot resolve {AP_REF!r}')
    peeled = [line for line in refs if line.endswith('^{}')]
    revision = (peeled or refs)[0].split()[0]
    source_record = dict(repository=repo, requested_ref=AP_REF, commit=revision)
    revision_file.write_text(json.dumps(source_record, indent=2))
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', '--no-checkout', repo, str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', 'origin', revision], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', revision], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'submodule', 'update', '--init', '--recursive'], check=True)
print('Installing AlphaPulldown AF3 dependencies…', flush=True)
ap3_run(['python', '-m', 'pip', 'install', '-v', str(SOURCE_DIR)+'[alphafold3]', 'setuptools<82'])
print('Building the pinned AF3 package…', flush=True)
ap3_run(['python', '-m', 'pip', 'install', '-v', '--no-deps', '-e', str(SOURCE_DIR/'alphafold3')])
ap3_run(['build_data'])
ap3_run(['python', '-m', 'pip', 'check'])
freeze = ap3_run(['python', '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
(RUN_DIR / 'environment_latest.txt').write_text(freeze)
source_record['submodules'] = subprocess.check_output(['git', '-C', str(SOURCE_DIR),
                                                      'submodule', 'status', '--recursive'], text=True)
(RUN_DIR / 'source_build.json').write_text(json.dumps(source_record, indent=2))
print('AF3 installation completed.', flush=True)


## 3. Verify the GPU inside the prediction environment
This performs a small JAX calculation on the GPU, rather than checking only whether Colab reports a GPU.


In [ ]:
subprocess.run(['nvidia-smi'], check=True)
ap3_run(['python', '-u', '-c', "import sys, jax, pdbfixer, alphafold3.cpp; import jax.numpy as jnp; "
        "import alphapulldown.folding_backend.alphafold3_backend; "
        "print('Python:', sys.executable); print('JAX:',jax.__version__); "
        "print('AF3 extension:',alphafold3.cpp.__file__); "
        "devices=jax.devices('gpu'); print(devices); "
        "x=jax.device_put(jnp.ones((32,32)),devices[0]); print((x@x).block_until_ready()[0,0])"])


## 4. Download and verify AF3 parameters
Downloads `af3.bin.zst` directly from DeepMind into a persistent Drive cache, then copies it to local disk for inference. See [official parameter download and terms](https://github.com/google-deepmind/alphafold3#obtaining-model-parameters). You can instead place your existing official weights at `WEIGHTS_CACHE/af3.bin.zst`. The loader supports this compressed format directly.


In [ ]:
WEIGHTS_CACHE.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
cached_weights = WEIGHTS_CACHE / 'af3.bin.zst'
if not cached_weights.exists():
    partial = WEIGHTS_CACHE / 'af3.bin.zst.partial'
    subprocess.run(['wget', '-c', '-O', str(partial),
                    'https://storage.googleapis.com/alphafold3/af3.bin.zst'], check=True)
    partial.replace(cached_weights)
if cached_weights.stat().st_size == 0:
    raise RuntimeError('Empty AF3 weight file.')
local_weights = MODEL_DIR / 'af3.bin.zst'
if not local_weights.exists() or local_weights.stat().st_size != cached_weights.stat().st_size:
    print('Copying weights to local disk…', flush=True)
    shutil.copy2(cached_weights, local_weights)
h = hashlib.sha256()
with local_weights.open('rb') as handle:
    for block in iter(lambda: handle.read(4*1024*1024), b''): h.update(block)
WEIGHT_SHA256 = h.hexdigest()
print('Loading AF3 weights to verify the file…', flush=True)
ap3_run(['python', '-u', '-c', "from alphafold3.model import params; "
        f"p=params.get_model_haiku_params(model_dir={str(MODEL_DIR)!r}); "
        "print('Verified weight scopes:',len(p))"])
print('Weight SHA256:', WEIGHT_SHA256)


## 5. Validate features and save a stable job manifest
Only load pickles you generated or obtained from a trusted source. This cell checks every requested object in the installed environment, hashes features for provenance, and records pair order. The manifest supports whole proteins, including bait homodimers if a name occurs in both lists. It does not generate all-versus-all pairs.


In [ ]:
import re, itertools, csv

def read_names(path):
    if not path.is_file():
        raise FileNotFoundError(f'Create or correct the protein list: {path}')
    lines = [line.strip() for line in path.read_text().splitlines()
             if line.strip() and not line.lstrip().startswith('#')]
    if any(line.startswith('>') for line in lines):
        if not lines[0].startswith('>'):
            raise ValueError(f'Mixed FASTA/list format in {path}')
        names = []
        for line in lines:
            if not line.startswith('>'):
                continue  # Sequence lines are not protein identifiers.
            header = line[1:].strip()
            if not header:
                raise ValueError(f'Empty FASTA header in {path}')
            name = header.split()[0]
            if name.startswith(('sp|', 'tr|')):
                name = name.split('|')[1]
            names.append(name)
    else:
        names = lines
    if not names:
        raise ValueError(f'Empty list: {path}')
    if len(names) != len(set(names)):
        raise ValueError(f'Duplicate identifiers in {path}')
    for name in names:
        if not re.fullmatch(r'[A-Za-z0-9_.-]+', name):
            raise ValueError(f'Use simple whole-protein feature identifiers: {name!r}')
    return names

def digest_file(path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

baits, preys = read_names(BAIT_LIST), read_names(PREY_LIST)
feature_paths = {}
for name in dict.fromkeys(baits + preys):
    options = [FEATURE_DIR / (name + suffix) for suffix in ('.pkl', '.pkl.xz')]
    matches = [p for p in options if p.is_file()]
    if len(matches) != 1:
        raise ValueError(f'{name}: expected exactly one .pkl or .pkl.xz in {FEATURE_DIR}; found {matches}')
    feature_paths[name] = matches[0]

validator = Path('/content/validate_ap_features.py')
validator.write_text("""import json, sys, pickle, lzma
items=json.loads(sys.argv[1]); paired=sys.argv[2]=='True'
for name,path in items.items():
    opener=lzma.open if path.endswith('.xz') else open
    with opener(path,'rb') as handle: obj=pickle.load(handle)
    if not hasattr(obj,'feature_dict') or not hasattr(obj,'sequence'):
        raise TypeError(f'{name}: expected an AlphaPulldown MonomericObject, not raw features/MSA')
    if obj.description != name:
        raise ValueError(f'{name}: object description is {obj.description!r}; names must match')
    for key in ('aatype','residue_index','msa'):
        if key not in obj.feature_dict: raise ValueError(f'{name}: missing {key}')
    if paired and (getattr(obj,'skip_msa',False) or 'msa_all_seq' not in obj.feature_dict):
        raise ValueError(f'{name}: pairing features missing; regenerate features or intentionally set PAIR_MSA=False')
    print(f'{name}: {len(obj.sequence)} residues, MSA shape {obj.feature_dict["msa"].shape}')
""")
ap3_run(['python', str(validator), json.dumps({k:str(v) for k,v in feature_paths.items()}), str(PAIR_MSA)])
settings = dict(backend='alphafold3', source_commit=revision, submodules=source_record['submodules'],
                weight_sha256=WEIGHT_SHA256, recycles=NUM_RECYCLES, seeds=NUM_SEEDS,
                diffusion_samples=DIFFUSION_SAMPLES, attention=FLASH_ATTENTION, base_seed=BASE_SEED,
                baits=baits, preys=preys,
                feature_sha256={k:digest_file(v) for k,v in feature_paths.items()})
manifest_path = RUN_DIR / 'run_settings.json'
if manifest_path.exists() and json.loads(manifest_path.read_text()) != settings:
    raise RuntimeError('Run inputs/settings changed. Choose a new RUN_NAME to avoid mixing predictions.')
manifest_path.write_text(json.dumps(settings, indent=2))
fingerprint = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
jobs = []
for index, (bait, prey) in enumerate(itertools.product(baits, preys), 1):
    job_id = f'{index:05d}_' + hashlib.sha256(f'{bait}+{prey}'.encode()).hexdigest()[:12]
    jobs.append(dict(index=index, bait=bait, prey=prey, job_id=job_id))
with (RUN_DIR / 'jobs.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['index','bait','prey','job_id'])
    writer.writeheader(); writer.writerows(jobs)
print(f'{len(baits)} bait(s) × {len(preys)} prey(s) = {len(jobs)} jobs')
print(f'{NUM_SEEDS * DIFFUSION_SAMPLES} AF3 samples per job')
print('First jobs:', jobs[:5])


## 6. Run the selected pairs
Uses the same AF2 objects, translating their alignments and templates for AF3. The first run tests one pair. Set `END_JOB=None` to screen all entries. Each pair runs in a separate process and logs to Drive. The compilation cache persists locally within a runtime; exact throughput depends on compilation and process overhead.

Completion requires the expected sample count, each sample's CIF and confidence JSON, and a finished command. Interrupted pairs rerun entirely. CIF is AF3's native structure output; optional AlphaPulldown metadata enrichment is disabled to avoid legacy feature-metadata conflicts.


In [ ]:
if not isinstance(START_JOB, int) or not 1 <= START_JOB <= len(jobs):
    raise ValueError('START_JOB must be within the manifest, starting at 1.')
last_job = len(jobs) if END_JOB is None else END_JOB
if not isinstance(last_job, int) or not START_JOB <= last_job <= len(jobs):
    raise ValueError('END_JOB must be None or within the manifest and >= START_JOB.')
expected_models = NUM_SEEDS * DIFFUSION_SAMPLES
failures = []
for job in jobs[START_JOB - 1:last_job]:
    out = RUN_DIR / 'predictions' / job['job_id']
    out.mkdir(parents=True, exist_ok=True)
    marker = out / 'colab_complete.json'
    def results_complete():
        rankings = list(out.rglob('ranking_scores.csv'))
        if len(rankings) != 1: return False
        try:
            with rankings[0].open() as handle: rows = list(csv.DictReader(handle))
            if len(rows) != expected_models: return False
            for row in rows:
                sample = rankings[0].parent / f"seed-{row['seed']}_sample-{row['sample']}"
                if not (sample/'model.cif').is_file() or (sample/'model.cif').stat().st_size == 0: return False
                for name in ('summary_confidences.json','confidences.json'):
                    if not isinstance(json.loads((sample/name).read_text()),dict): return False
            return True
        except (OSError,ValueError,KeyError,TypeError): return False
    if marker.exists() and json.loads(marker.read_text()).get('fingerprint') == fingerprint and results_complete():
        print(f"Skip completed {job['index']}: {job['bait']} + {job['prey']}")
        continue
    args = ['run_structure_prediction.py', f"--input={job['bait']}+{job['prey']}",
            f'--features_directory={FEATURE_DIR}', f'--data_directory={MODEL_DIR}',
            f'--output_directory={out}', '--fold_backend=alphafold3',
            '--protein_delimiter=+', '--use_ap_style=False', '--convert_to_modelcif=False',
            f'--num_recycles={NUM_RECYCLES}', f'--num_seeds={NUM_SEEDS}',
            f'--num_diffusion_samples={DIFFUSION_SAMPLES}',
            f'--flash_attention_implementation={FLASH_ATTENTION}',
            '--jax_compilation_cache_dir=/content/af3_jax_cache',
            f"--random_seed={BASE_SEED + job['index']}"]
    command = [CONDA, 'run', '--no-capture-output', '-p', str(ENV_PREFIX), *args]
    print(f"Run {job['index']}/{len(jobs)}: {job['bait']} + {job['prey']}", flush=True)
    started = time.time()
    with (out / 'prediction.log').open('a') as log:
        log.write('\nCommand: ' + repr(command) + '\n'); log.flush()
        process = subprocess.Popen(command, env=RUNTIME_ENV, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if returncode != 0 or not results_complete():
        failures.append(job['index'])
        print('Failed or incomplete; inspect', out / 'prediction.log')
        if not CONTINUE_ON_ERROR:
            raise RuntimeError(f"Job {job['index']} failed; rerun after resolving its log error.")
        continue
    marker.write_text(json.dumps(dict(fingerprint=fingerprint, job=job,
                                      seconds=time.time()-started), indent=2))
print('Finished selected range. Failed jobs:', failures)


## 7. Export AF3 scores
Writes one row per sample and a best-ranked row per pair. AF3 ranking score includes terms beyond ipTM/pTM; compare AF3 and AF2 scores separately. Inspect interface PAE, contacts and repeat-seed consistency before treating a candidate as convincing.


In [ ]:
rows = []
for job in jobs:
    out = RUN_DIR/'predictions'/job['job_id']
    for ranking in out.rglob('ranking_scores.csv'):
        with ranking.open() as handle: samples = list(csv.DictReader(handle))
        for sample in samples:
            folder = ranking.parent/f"seed-{sample['seed']}_sample-{sample['sample']}"
            path = folder/'summary_confidences.json'
            if not path.exists(): continue
            summary = json.loads(path.read_text())
            rows.append(dict(job_index=job['index'],bait=job['bait'],prey=job['prey'],
                             seed=sample['seed'],sample=sample['sample'],
                             ranking_score=float(sample['ranking_score']),
                             iptm=summary.get('iptm'),ptm=summary.get('ptm'),
                             fraction_disordered=summary.get('fraction_disordered'),
                             has_clash=summary.get('has_clash'),
                             complete=(out/'colab_complete.json').exists(),
                             structure=str(folder/'model.cif')))
fields=['job_index','bait','prey','seed','sample','ranking_score','iptm','ptm',
        'fraction_disordered','has_clash','complete','structure']
def save_rows(name, records):
    with (RUN_DIR/name).open('w',newline='') as handle:
        writer=csv.DictWriter(handle,fieldnames=fields); writer.writeheader(); writer.writerows(records)
save_rows('af3_sample_scores.csv',rows)
best={}
for row in rows:
    if not row['complete']: continue
    key=row['job_index']
    if key not in best or row['ranking_score'] > best[key]['ranking_score']: best[key]=row
save_rows('af3_best_per_pair.csv',sorted(best.values(),key=lambda r:r['ranking_score'],reverse=True))
print(f'Saved {len(rows)} sample rows and {len(best)} completed pairs in {RUN_DIR}')


## Notes and troubleshooting
- This notebook requires trusted AlphaPulldown monomer pickles, not raw A3M files. Headers/lists must match object descriptions; FASTA inputs are accepted by the reader.
- Use a fresh GPU runtime for AF3. If an installation/build fails, retain the verbose output for diagnosis; do not independently upgrade JAX.
- Existing paired-MSA fields are validated. Reusing AF2 features is not equivalent to running AF3's native database searches.
- Each pair's prepared `*_data.json` documents the translated inputs. Both CIF structures and PAE/confidence JSON files are retained.
- After a runtime reset reinstall from the saved commit, reload weights, then rerun the manifest and prediction cells. Completed pairs are skipped; incomplete pairs restart.
- Choose a new `RUN_NAME` for changed input sequences, feature files, settings or source revision. Do not run two sessions writing the same run folder.
- AF3 runs in an independent environment. For missing AF2 features use the AF2 notebook's remote MMseqs2 feature generator, then return here.

References: [AlphaPulldown AF3 installation](https://github.com/KosinskiLab/AlphaPulldown/wiki/Run-AlphaPulldown-Python-Command-Line-Interface), [AF3 backend](https://github.com/KosinskiLab/AlphaPulldown/blob/main/alphapulldown/folding_backend/alphafold3_backend.py), [official AF3 weights](https://github.com/google-deepmind/alphafold3#obtaining-model-parameters).
